In [1]:
!pip install azure-ai-ml azure-identity scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 32.6 MB/s  0:00:00m0:00:010:02
  Attempting uninstall: psutil
    Found existing installation: psutil 5.2.2
    Uninstalling psutil-5.2.2:
      Successfully uninstalled psutil-5.2.2
  Attempting uninstall: opentelemetry-api━━━━━━━━━━━━━━━━━━━━━━━━━  5/29 [strictyaml]
    Found existing installation: opentelemetry-api 1.33.0━━━━━  5/29 [strictyaml]
    Uninstalling opentelemetry-api-1.33.0:━━━━━━━━━━━━━━━━━━━━  5/29 [strictyaml]
      Successfully uninstalled opentelemetry-api-1.33.0━━━━━━━━━━━  6/29 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions━━━━  6/29 [opentelemetry-api]
    Found existing installation: opentelemetry-semantic-conventions 0.54b0/29 [opentelemetry-api]
    Uninstalling opentelemetry-semantic-conventions-0.54b0:━  7/29 [opentelemetry-semantic-conventions]
      Successfully uninstalled opentelemetry-semantic-conventions-0.54b0 [opentelemetry-semantic-conventions]
  Attempting uni

In [2]:
import pandas as pd
import numpy as np
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    CodeConfiguration,
    Data
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import requests
import json
import os

In [3]:
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'sqft':      np.random.randint(800, 5000, n),
    'bedrooms':  np.random.randint(1, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age':       np.random.randint(1, 50, n),
})

df['price'] = (
    df['sqft'] * 150 +
    df['bedrooms'] * 10000 +
    df['bathrooms'] * 8000 -
    df['age'] * 500 +
    np.random.randint(-20000, 20000, n)
)

df = df[['price', 'sqft', 'bedrooms', 'bathrooms', 'age']]
df.to_csv('train.csv', index=False)

print(f"Shape: {df.shape}")
print(df.head())
print("train.csv ready!")

Shape: (200, 5)
    price  sqft  bedrooms  bathrooms  age
0  283833  1660         4          1   33
1  733450  4572         3          1   20
2  625974  3892         1          3   13
3  237754  1266         3          2   28
4  626752  4244         1          2   48
train.csv ready!


In [9]:
# These values are from your Azure portal
subscription_id = "8f88523f-ab4d-425a-9e2f-2dd5877554ab"    # from portal.azure.com
resource_group  = "ml-azure"
workspace_name  = "adityaml"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name
)

print("Connected to:", workspace_name)

Connected to: adityaml


In [10]:
my_data = Data(
    path="train.csv",
    type=AssetTypes.URI_FILE,
    name="house-price-train",
    description="House price training data"
)

uploaded = ml_client.data.create_or_update(my_data)
print(f"Data uploaded to: {uploaded.path}")

Uploading train.csv (< 1 MB): 100%|██████████| 3.79k/3.79k [00:00<00:00, 257kB/s]




Data uploaded to: azureml://subscriptions/8f88523f-ab4d-425a-9e2f-2dd5877554ab/resourcegroups/ml-azure/workspaces/adityaml/datastores/workspaceblobstore/paths/LocalUpload/c8a871bbe4b956fcd0b39fd2c0de0c70a6b49a495be441b7c830dce45ef61a1e/train.csv


In [11]:
os.makedirs('src', exist_ok=True)

training_script = '''
import argparse
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib, os

parser = argparse.ArgumentParser()
parser.add_argument("--data",       type=str)
parser.add_argument("--output_dir", type=str, default="outputs")
args = parser.parse_args()

df = pd.read_csv(args.data)
X = df[["sqft","bedrooms","bathrooms","age"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)

rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f"RMSE: {rmse:.2f}")

os.makedirs(args.output_dir, exist_ok=True)
joblib.dump(model, os.path.join(args.output_dir, "model.pkl"))
print("Model saved!")
'''

with open('src/train.py', 'w') as f:
    f.write(training_script)

print("src/train.py created!")

src/train.py created!


In [13]:
job = command(
    code="./src",
    command="python train.py --data ${{inputs.training_data}} --output_dir ${{outputs.model_output}}",
    inputs={
        "training_data": Input(
            type=AssetTypes.URI_FILE,
            path=uploaded.path
        )
    },
    outputs={
        "model_output": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="azureml://registries/azureml/environments/sklearn-1.5/versions/1",
    compute="cluteradi",
    display_name="house-price-training",
    experiment_name="house-price-prediction"
)

returned_job = ml_client.jobs.create_or_update(job)
print(f"Job name: {returned_job.name}")
print(f"Track here: {returned_job.studio_url}")

# Wait for completion
ml_client.jobs.stream(returned_job.name)
print("Training complete!")

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


Job name: maroon_kite_sv41mtc6xl
Track here: https://ml.azure.com/runs/maroon_kite_sv41mtc6xl?wsid=/subscriptions/8f88523f-ab4d-425a-9e2f-2dd5877554ab/resourcegroups/ml-azure/workspaces/adityaml&tid=1cfd9168-d236-4130-9f36-48ae7f8df486
RunId: maroon_kite_sv41mtc6xl
Web View: https://ml.azure.com/runs/maroon_kite_sv41mtc6xl?wsid=/subscriptions/8f88523f-ab4d-425a-9e2f-2dd5877554ab/resourcegroups/ml-azure/workspaces/adityaml

Execution Summary
RunId: maroon_kite_sv41mtc6xl
Web View: https://ml.azure.com/runs/maroon_kite_sv41mtc6xl?wsid=/subscriptions/8f88523f-ab4d-425a-9e2f-2dd5877554ab/resourcegroups/ml-azure/workspaces/adityaml

Training complete!


In [14]:
model = Model(
    path=f"azureml://jobs/{returned_job.name}/outputs/model_output/model.pkl",
    name="house-price-model",
    description="House price linear regression",
    type=AssetTypes.CUSTOM_MODEL
)

registered_model = ml_client.models.create_or_update(model)
print(f"Model: {registered_model.name} v{registered_model.version}")

Model: house-price-model v1


In [15]:
scoring_script = '''
import joblib, json, os, numpy as np

def init():
    global model
    path = os.path.join(os.environ["AZUREML_MODEL_DIR"], "model.pkl")
    model = joblib.load(path)
    print("Model loaded!")

def run(raw_data):
    data = json.loads(raw_data)["data"]
    prediction = model.predict(np.array(data))
    return {"predictions": prediction.tolist()}
'''

with open('src/score.py', 'w') as f:
    f.write(scoring_script)

print("src/score.py created!")

src/score.py created!


In [16]:
# Create endpoint
endpoint = ManagedOnlineEndpoint(
    name="house-price-endpoint",
    auth_mode="key"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created!")

# Deploy model to endpoint
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="house-price-endpoint",
    model=registered_model.id,
    code_configuration=CodeConfiguration(
        code="./src",
        scoring_script="score.py"
    ),
    environment="azureml://registries/azureml/environments/sklearn-1.5/versions/1",
    instance_type="Standard_DS2_v2",
    instance_count=1
)
ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployed!")

# Route 100% traffic to this deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic routed!")

Endpoint created!
...................................................................................Deployed!
Traffic routed!


Uploading src (0.0 MBs): 100%|██████████| 1192/1192 [00:00<00:00, 19147.67it/s]




In [ ]:
endpoint_details = ml_client.online_endpoints.get("house-price-endpoint")
keys = ml_client.online_endpoints.get_keys("house-price-endpoint")

url    = endpoint_details.scoring_uri
apikey = keys.primary_key

# [sqft, bedrooms, bathrooms, age]
payload = json.dumps({"data": [[2500, 3, 2, 15]]})

headers = {
    "Content-Type":  "application/json",
    "Authorization": f"Bearer {apikey}"
}

response = requests.post(url, data=payload, headers=headers)
result   = response.json()
print(f"Predicted price: ${result['predictions'][0]:,.2f}")